# Data Quality Check

## Supply Chain Inventory & Demand Planning Analytics

This notebook profiles the raw supply-chain datasets, validates key data-quality and business rules, and creates clean processed datasets for downstream analysis.

**Workflow:** Raw Data → Profiling → Validation → Cleaning → Processed Data


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data: {RAW_DIR}")
print(f"Processed data: {PROCESSED_DIR}")


## 1. Load Raw Datasets

The raw CSV files are loaded directly from the `data/raw/` directory.


In [ ]:
suppliers = pd.read_csv(RAW_DIR / "suppliers.csv")
products = pd.read_csv(RAW_DIR / "products.csv")
inventory = pd.read_csv(RAW_DIR / "inventory.csv")
sales = pd.read_csv(RAW_DIR / "sales.csv")
purchase_orders = pd.read_csv(RAW_DIR / "purchase_orders.csv")

datasets = {
    "Suppliers": suppliers,
    "Products": products,
    "Inventory": inventory,
    "Sales": sales,
    "Purchase Orders": purchase_orders
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")


## 2. Dataset Structure and Data Types

Review the structure, columns, and data types before performing validation.


In [ ]:
for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * len(name))
    print("Columns:", list(df.columns))
    print("Shape:", df.shape)
    print(df.dtypes)


## 3. Missing-Value Assessment

Check every dataset for missing values.


In [ ]:
missing_summary = pd.DataFrame({
    name: df.isna().sum()
    for name, df in datasets.items()
}).fillna(0).astype(int)

display(missing_summary)


## 4. Duplicate-Row Check

Identify duplicate records that could distort downstream analysis.


In [ ]:
duplicate_summary = pd.Series({
    name: int(df.duplicated().sum())
    for name, df in datasets.items()
}, name="duplicate_rows")

display(duplicate_summary.to_frame())


## 5. Primary-Key Validation

Validate uniqueness of the main identifiers and the inventory composite key.


In [ ]:
pk_checks = {
    "Supplier ID unique": suppliers["supplier_id"].is_unique,
    "Product ID unique": products["product_id"].is_unique,
    "Sales Order ID unique": sales["order_id"].is_unique,
    "Purchase Order ID unique": purchase_orders["po_id"].is_unique,
    "Inventory composite key unique": not inventory.duplicated(
        subset=["date", "product_id", "warehouse_id"]
    ).any()
}

for check, result in pk_checks.items():
    print(f"{check}: {result}")


## 6. Inventory Reconciliation

Validate the inventory accounting relationship:

**Opening Stock + Receipts − Demand + Stockout Units = Closing Stock**

Stockout units are included because unmet demand is represented separately from physical closing inventory.


In [ ]:
inventory["calculated_closing_stock"] = (
    inventory["opening_stock"]
    + inventory["receipts"]
    - inventory["demand"]
)

reconciliation_errors = int(
    (
        inventory["calculated_closing_stock"]
        != inventory["closing_stock"]
    ).sum()
)

print(f"Inventory records: {len(inventory):,}")
print(f"Initial reconciliation errors: {reconciliation_errors:,}")


In [ ]:
inventory["expected_closing_stock"] = (
    inventory["opening_stock"]
    + inventory["receipts"]
    - inventory["demand"]
    + inventory["stockout_units"]
)

adjusted_errors = int(
    (
        inventory["expected_closing_stock"]
        != inventory["closing_stock"]
    ).sum()
)

print(f"Adjusted reconciliation errors: {adjusted_errors:,}")
print(
    f"Adjusted reconciled records: "
    f"{len(inventory) - adjusted_errors:,} / {len(inventory):,}"
)


### Inventory Reconciliation Finding

The initial reconciliation difference corresponds to records containing stockout units. After incorporating stockout units into the inventory equation, all inventory records reconcile successfully.


## 7. Negative-Value Validation

Inventory quantities should not contain negative values.


In [ ]:
negative_checks = {
    "opening_stock": int((inventory["opening_stock"] < 0).sum()),
    "receipts": int((inventory["receipts"] < 0).sum()),
    "demand": int((inventory["demand"] < 0).sum()),
    "closing_stock": int((inventory["closing_stock"] < 0).sum()),
    "stockout_units": int((inventory["stockout_units"] < 0).sum())
}

for column, count in negative_checks.items():
    print(f"{column}: {count}")


## 8. Sales Business-Rule Validation

Validate ordered quantities, fulfilled quantities, and unfulfilled demand.


In [ ]:
sales["unfulfilled_quantity"] = (
    sales["quantity_ordered"]
    - sales["quantity_fulfilled"]
)

negative_sales_quantities = {
    "quantity_ordered": int((sales["quantity_ordered"] < 0).sum()),
    "quantity_fulfilled": int((sales["quantity_fulfilled"] < 0).sum())
}

fulfilled_exceeds_ordered = int(
    (
        sales["quantity_fulfilled"]
        > sales["quantity_ordered"]
    ).sum()
)

total_ordered = sales["quantity_ordered"].sum()
total_fulfilled = sales["quantity_fulfilled"].sum()
total_unfulfilled = sales["unfulfilled_quantity"].sum()

print("Negative quantity checks:")
for column, count in negative_sales_quantities.items():
    print(f"  {column}: {count}")

print(f"Fulfilled > ordered errors: {fulfilled_exceeds_ordered:,}")
print(f"Total ordered: {total_ordered:,}")
print(f"Total fulfilled: {total_fulfilled:,}")
print(f"Total unfulfilled: {total_unfulfilled:,}")
print(f"Fill rate: {total_fulfilled / total_ordered:.2%}")


## 9. Purchase-Order Date and Lead-Time Validation

Validate purchase-order dates and calculate actual supplier lead time.


In [ ]:
date_columns = [
    "order_date",
    "expected_date",
    "actual_date"
]

for column in date_columns:
    purchase_orders[column] = pd.to_datetime(
        purchase_orders[column],
        errors="coerce"
    )

invalid_dates = {
    column: int(purchase_orders[column].isna().sum())
    for column in date_columns
}

purchase_orders["calculated_lead_time_days"] = (
    purchase_orders["actual_date"]
    - purchase_orders["order_date"]
).dt.days

negative_lead_times = int(
    (purchase_orders["calculated_lead_time_days"] < 0).sum()
)

print("Invalid dates:")
for column, count in invalid_dates.items():
    print(f"  {column}: {count}")

print(f"Negative lead times: {negative_lead_times:,}")
print("\nLead-time statistics:")
display(
    purchase_orders["calculated_lead_time_days"]
    .describe()
)


## 10. Stockout Validation

Confirm that stockout records correspond to zero closing inventory.


In [ ]:
stockout_records = int(
    (inventory["stockout_units"] > 0).sum()
)

zero_stock_records = int(
    (inventory["closing_stock"] == 0).sum()
)

stockout_mismatch = int(
    (
        (inventory["stockout_units"] > 0)
        != (inventory["closing_stock"] == 0)
    ).sum()
)

print(
    f"Records with stockout units > 0: "
    f"{stockout_records:,}"
)

print(
    f"Records with closing stock = 0: "
    f"{zero_stock_records:,}"
)

print(
    f"Stockout logic mismatches: "
    f"{stockout_mismatch:,}"
)


## 11. Create Processed Datasets

Reload the original raw files so temporary validation columns are not included in the processed datasets.

Dates are standardized to datetime before export.


In [ ]:
suppliers_clean = pd.read_csv(
    RAW_DIR / "suppliers.csv"
)

products_clean = pd.read_csv(
    RAW_DIR / "products.csv"
)

inventory_clean = pd.read_csv(
    RAW_DIR / "inventory.csv"
)

sales_clean = pd.read_csv(
    RAW_DIR / "sales.csv"
)

purchase_orders_clean = pd.read_csv(
    RAW_DIR / "purchase_orders.csv"
)

inventory_clean["date"] = pd.to_datetime(
    inventory_clean["date"]
)

sales_clean["order_date"] = pd.to_datetime(
    sales_clean["order_date"]
)

for column in [
    "order_date",
    "expected_date",
    "actual_date"
]:
    purchase_orders_clean[column] = pd.to_datetime(
        purchase_orders_clean[column]
    )

clean_datasets = {
    "suppliers_clean.csv": suppliers_clean,
    "products_clean.csv": products_clean,
    "inventory_clean.csv": inventory_clean,
    "sales_clean.csv": sales_clean,
    "purchase_orders_clean.csv": purchase_orders_clean
}

for filename, df in clean_datasets.items():
    df.to_csv(
        PROCESSED_DIR / filename,
        index=False
    )

    print(
        f"Saved: {filename} "
        f"({len(df):,} rows, {len(df.columns)} columns)"
    )


## 12. Processed Dataset Validation

Confirm that the exported processed datasets have the expected dimensions.


In [ ]:
for filename, df in clean_datasets.items():
    print(
        f"{filename}: "
        f"{df.shape[0]:,} rows × {df.shape[1]} columns"
    )


## 13. Data Quality Summary

### Validation Results

- Missing values: checked across all five datasets.
- Duplicate rows: checked across all five datasets.
- Primary keys: validated for uniqueness.
- Inventory reconciliation: validated using stockout-adjusted accounting logic.
- Negative inventory quantities: checked.
- Sales quantities: validated against business rules.
- Purchase-order dates: validated and converted to datetime.
- Supplier lead time: calculated from order date to actual delivery date.
- Stockout records: validated against zero closing inventory.
- Processed datasets: exported to `data/processed/`.

The processed datasets are now ready for exploratory analysis, inventory analytics, supplier analysis, and demand forecasting.
